# Model 3: Random Forest Classifier
## IoT Saldırı Tipi Sınıflandırması (Multi-class)

Bu notebook, **Edge-IIoTset** veri seti üzerinde **Attack_type** kolonunu hedef değişken olarak kullanan
Random Forest (Rastgele Orman) modelini adım adım eğitir ve değerlendirir.

### Neden Random Forest?
- **Ensemble (Bagging)**: Birden fazla karar ağacının oy çokluğu ile karar verir → daha kararlı tahminler
- **Overfitting'e dayanıklı**: Tek ağaca göre genelleme gücü daha yüksek
- **Feature importance**: Her ağacın katkısını birleştirerek güvenilir önem skoru verir
- **Paralelleştirilebilir**: Her ağaç bağımsız eğitilir

### Pipeline Adımları
1. Gold Delta Lake → Veri yükleme
2. Feature vektörleme + StringIndexer
3. Sınıf ağırlıkları (classWeight)
4. Stratified Train/Test split (%80/%20)
5. RandomForestClassifier
6. CrossValidator ile hiperparametre arama (numTrees, maxDepth, minInstancesPerNode)
7. Test seti değerlendirme
8. Orman yapı analizi + Feature importance
9. MLflow loglama

## 1. Kütüphaneler ve Spark Session

In [ ]:
import sys
import time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

sys.path.insert(0, "/opt/bitnami/spark")

from spark.spark_session import get_spark
from ml.utils import (
    run_ml_pipeline_multiclass,
    evaluate_model_multiclass,
    compute_confusion_matrix_multiclass,
    log_to_mlflow,
)

print("Kütüphaneler yüklendi.")

In [ ]:
spark = get_spark("Notebook-RandomForest-Multiclass")
print(f"Spark version: {spark.version}")
print(f"App name: {spark.sparkContext.appName}")

## 2. Veri Yükleme ve Feature Hazırlığı

In [ ]:
SAMPLE_SIZE = 30000  # Hızlı test için (None = tüm veri)

stage_start = time.perf_counter()
train_df, test_df, feature_cols, label_index_model = run_ml_pipeline_multiclass(
    spark,
    sample_size=SAMPLE_SIZE,
    split_log_stats=True,
)
label_names = list(label_index_model.labels)
num_classes = len(label_names)

print(f"\nPipeline süresi: {time.perf_counter() - stage_start:.2f}s")
print(f"Feature sayısı: {len(feature_cols)}")
print(f"Sınıf sayısı: {num_classes}")
print(f"Sınıflar: {label_names}")

In [ ]:
train_count = train_df.count()
test_count = test_df.count()
print(f"Train: {train_count:,} satır")
print(f"Test:  {test_count:,} satır")

print("\nTrain seti sınıf dağılımı:")
train_df.groupBy("label").count().orderBy("label").show()

## 3. Model Pipeline Oluşturma

Random Forest ağaç tabanlı olduğundan scaling gerekmez.

### Hiperparametreler
| Parametre | Açıklama | Aranacak Değerler |
|-----------|----------|------------------|
| `numTrees` | Ormandaki ağaç sayısı | 50, 100 |
| `maxDepth` | Her ağacın maksimum derinliği | 10, 15 |
| `minInstancesPerNode` | Yaprak düğüm min örnek | 1, 5 |

In [ ]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    numTrees=100,
    maxDepth=15,
    seed=42,
)
pipeline = Pipeline(stages=[rf])
print("Pipeline oluşturuldu: RandomForestClassifier")

## 4. Cross Validation

In [ ]:
num_trees_values = [50, 100]
max_depth_values = [10, 15]
min_instances_values = [1, 5]
num_folds = 3

param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, num_trees_values)
    .addGrid(rf.maxDepth, max_depth_values)
    .addGrid(rf.minInstancesPerNode, min_instances_values)
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1",
    ),
    numFolds=num_folds,
    seed=42,
    parallelism=2,
)

total_cv_runs = len(param_grid) * num_folds
print(f"Grid boyutu: {len(param_grid)} kombinasyon")
print(f"Fold sayısı: {num_folds}")
print(f"Toplam fit: {total_cv_runs}")

## 5. Model Eğitimi

In [ ]:
print("Cross Validation başlatılıyor...")
cv_start = time.perf_counter()
cv_model = cv.fit(train_df)
cv_duration = time.perf_counter() - cv_start
print(f"CV tamamlandı! Süre: {cv_duration:.2f}s")

best_rf_model = cv_model.bestModel.stages[-1]
print(f"\nEn iyi parametreler:")
print(f"  numTrees:            {best_rf_model.getNumTrees}")
print(f"  maxDepth:            {best_rf_model.getOrDefault('maxDepth')}")
print(f"  minInstancesPerNode: {best_rf_model.getOrDefault('minInstancesPerNode')}")

In [ ]:
# CV sonuçları
print("CV Sonuçları (F1-Score):")
print(f"{'#':<4} {'Parametreler':<45} {'Avg F1':>10}")
print("-" * 62)
for i, (params, score) in enumerate(zip(param_grid, cv_model.avgMetrics)):
    param_str = ", ".join(f"{p.name}={v}" for p, v in params.items())
    marker = " ← best" if score == max(cv_model.avgMetrics) else ""
    print(f"{i+1:<4} {param_str:<45} {score:>10.4f}{marker}")

## 6. Test Seti Değerlendirme

In [ ]:
predictions = cv_model.bestModel.transform(test_df)
metrics = evaluate_model_multiclass(predictions, num_classes=num_classes)

In [ ]:
confusion = compute_confusion_matrix_multiclass(predictions, label_names=label_names)

In [ ]:
print("\n" + "=" * 40)
print("SONUÇ ÖZETİ")
print("=" * 40)
print(f"Accuracy:  {metrics.get('accuracy', 0):.4f}")
print(f"F1-Score:  {metrics.get('f1_score', 0):.4f}")
print(f"Precision: {metrics.get('precision', 0):.4f}")
print(f"Recall:    {metrics.get('recall', 0):.4f}")

## 7. Orman Yapı Analizi

In [ ]:
# Orman yapı bilgileri
num_trees = best_rf_model.getNumTrees
total_nodes = best_rf_model.totalNumNodes
depths = [t.depth for t in best_rf_model.trees]
nodes_per_tree = [t.numNodes for t in best_rf_model.trees]
avg_depth = sum(depths) / len(depths) if depths else 0

print(f"Ağaç Sayısı:      {num_trees}")
print(f"Toplam Düğüm:     {total_nodes}")
print(f"Ort. Derinlik:    {avg_depth:.1f}")
print(f"Min Derinlik:     {min(depths) if depths else 0}")
print(f"Max Derinlik:     {max(depths) if depths else 0}")

In [ ]:
# Ağaç derinlikleri dağılımı
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(depths, bins=max(10, len(set(depths))), color="#4CAF50", edgecolor="#2E7D32", alpha=0.8)
axes[0].set_xlabel("Derinlik", fontsize=11)
axes[0].set_ylabel("Ağaç Sayısı", fontsize=11)
axes[0].set_title("Ağaç Derinlikleri Dağılımı", fontsize=13, fontweight="bold")
axes[0].axvline(avg_depth, color="red", linestyle="--", label=f"Ort: {avg_depth:.1f}")
axes[0].legend()

axes[1].hist(nodes_per_tree, bins=max(10, len(set(nodes_per_tree))), color="#66BB6A", edgecolor="#2E7D32", alpha=0.8)
axes[1].set_xlabel("Düğüm Sayısı", fontsize=11)
axes[1].set_ylabel("Ağaç Sayısı", fontsize=11)
axes[1].set_title("Ağaç Düğüm Sayıları Dağılımı", fontsize=13, fontweight="bold")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
importances = best_rf_model.featureImportances.toArray().tolist()
fi_pairs = sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True)
top_features = fi_pairs[:15]

print("Top 15 Feature Importance:")
print(f"{'#':<4} {'Feature':<35} {'Importance':>12}")
print("-" * 55)
for idx, (fname, importance) in enumerate(top_features, start=1):
    print(f"{idx:<4} {fname:<35} {importance:>12.6f}")

In [ ]:
# Feature Importance Görselleştirmesi
top10 = fi_pairs[:10]
names = [f[0] for f in reversed(top10)]
values = [f[1] for f in reversed(top10)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, values, color="#4CAF50", edgecolor="#2E7D32", height=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9, fontweight="bold")

ax.set_xlabel("Feature Importance", fontsize=11)
ax.set_title("Random Forest — Top 10 Feature Importance (Multi-class)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 9. MLflow'a Loglama

In [ ]:
best_params = {
    "numTrees": int(best_rf_model.getNumTrees),
    "maxDepth": int(best_rf_model.getOrDefault("maxDepth")),
    "minInstancesPerNode": int(best_rf_model.getOrDefault("minInstancesPerNode")),
    "numClasses": int(num_classes),
    "numFolds": num_folds,
    "grid_size": len(param_grid),
    "cv_total_fits": total_cv_runs,
    "weightCol": "classWeight",
    "forest_total_nodes": total_nodes,
    "forest_avg_depth": round(avg_depth, 2),
    "label_column": "Attack_type",
}

cv_best_f1 = float(max(cv_model.avgMetrics))
metrics["cv_best_f1"] = cv_best_f1
metrics["forest_num_trees"] = float(num_trees)
metrics["forest_total_nodes"] = float(total_nodes)
metrics["forest_avg_depth"] = float(avg_depth)

confusion_metrics = {}
for i, name in enumerate(label_names):
    confusion_metrics[f"row_total_class_{i}"] = confusion["row_totals"][i]
    confusion_metrics[f"per_class_acc_{i}"] = confusion["per_class_acc"][i]

run_id = log_to_mlflow(
    run_name="random_forest_notebook",
    model_type="RandomForest",
    params=best_params,
    metrics={**metrics, **confusion_metrics},
    model=cv_model.bestModel,
    feature_importance=top_features[:10],
    tags={
        "source": "notebook",
        "model_index": "3",
        "classification_type": "multiclass",
        "ensemble_method": "bagging",
    },
)
print(f"\nMLflow Run ID: {run_id}")

## Özet

Bu notebook'ta **Random Forest** modelini başarıyla eğittik:

- **Bagging (Bootstrap Aggregating)** yöntemi ile birden fazla ağaç eğitildi
- **CrossValidator** ile numTrees, maxDepth, minInstancesPerNode optimize edildi
- **classWeight** ile sınıf dengesizliği telafi edildi
- Orman yapısı (ağaç sayısı, derinlik dağılımı) analiz edildi
- **Feature importance** tüm ağaçların katkısıyla hesaplandı
- Sonuçlar **MLflow**'a loglandı

Random Forest, Decision Tree'nin genelleştirme gücünü artıran güçlü bir ensemble yöntemidir.

In [ ]:
spark.stop()
print("Spark oturumu kapatıldı.")